## Users UA 
    * Geocode locations with missing coordinates
    * Spatial merge with census data at the district level 
    * Identity proxies: based on name, based on language overall (name, reported location)

In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv
import geopandas as gpd
import numpy as np
import os, sys, re
from os import listdir
from os.path import isfile, join
import datetime as dt
import matplotlib.pyplot as plt

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

In [ ]:
users = pd.read_csv(EXTRACTIONS / 'ua_users.csv', sep = '|', encoding = 'utf-8')

In [ ]:
len(users['id'].unique())

In [ ]:
for i in ['lat', 'long']: 
    users[f'{i}_current']=users[f'{i}_current'].str.replace(',', '')
    users[f'{i}_current']=users[f'{i}_current'].str.replace('\\', '')
    users[f'{i}_current']=users[f'{i}_current'].str.replace('n', '')
    users.loc[(users[f'{i}_current']=='') | (users[f'{i}_current']== "NA"), f'{i}_current'] = 0
    users[f'{i}_current']=users[f'{i}_current'].astype(float)

In [ ]:
users_df = gpd.GeoDataFrame(users, geometry=gpd.points_from_xy(users.long_current, users.lat_current), crs="EPSG:4326")

In [ ]:
users_df_cl = users_df[users_df.is_valid].reset_index(drop = True)

In [ ]:
#https://juanitorduz.github.io/germany_plots/
ua_df = gpd.read_file(EXTRACTIONS / 'UKR_adm_shp/UKR_adm1.shp')

In [ ]:
ua_data = pd.read_stata('database/stata_tmp/ukraine_regions.dta')

In [ ]:
# merge on ID_1 = code_geo
ua_df = pd.merge(
    left=ua_df, 
    right=ua_data, 
    left_on='ID_1', right_on = 'code_geo',
    how='left'
)
ua_df

In [ ]:
# add centroid
ua_df["center"] = ua_df["geometry"].centroid
points = ua_df.copy()
points.set_geometry("center", inplace = True)

In [ ]:
fig, ax = plt.subplots(figsize=(25,10))

#ua_df.plot(ax=ax, color='lemonchiffon', alpha=0.8)
ua_df.plot(ax=ax, column = 'language_ru_T', categorical=False, legend=False,  
                                      cmap = 'Blues', alpha=1, vmin = 0, vmax = 100, 
          legend_kwds={'label': "Share of population with Russian as the native language",
                        'orientation': "horizontal"})
ua_df.boundary.plot(ax=ax, linewidth = 0.5)
texts =[]
for x, y, code, label in zip(points.geometry.x, points.geometry.y, points.code_geo, points["language_ru_T"].astype(float).round(2)):
    if code == 11:
        plt.text(x, y, label, fontsize = 12, ha = 'left', va = 'top')
    else:
        plt.text(x, y, label, fontsize = 12, ha = 'right', va = 'top')

# add name of a capital
for x, y, code, label in zip(points.geometry.x, points.geometry.y, points.code_geo, points.NAME_1):
    if code == 11:
        plt.text(x, y, label, fontsize = 12, va = 'bottom', ha = 'left')
    else:
        plt.text(x, y, label, fontsize = 12, ha = 'right', va = 'bottom')

ax.set(
    title = '',
    #title='Russian as the first language, share of population', 
    aspect=1.3, 
    facecolor='lemonchiffon'
);
fig.savefig(FIGURES / "map_ru_lang.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(25,10))

#ua_df.plot(ax=ax, color='lemonchiffon', alpha=0.8)
ua_df.plot(ax=ax, column = 'russians', categorical=False, legend=False,  
                                      cmap = 'Blues', alpha=1, vmin = 0, vmax = 70)
ua_df.boundary.plot(ax=ax, linewidth = 0.5)
texts =[]
for x, y, code, label in zip(points.geometry.x, points.geometry.y, points.code_geo, points["russians"].astype(float).round(2)):
    if code == 11:
        plt.text(x, y, label, fontsize = 12, ha = 'center')
    else:
        plt.text(x, y, label, fontsize = 12, ha = 'right')
ax.set(
    title = '',
    #title='Russian as the first language, share of population', 
    aspect=1.3, 
    facecolor='lemonchiffon'
);
fig.savefig(FIGURES / "map_russians.pdf", bbox_inches='tight')

In [ ]:
# spatial match with geocoded users 
users_codes = users_df_cl.sjoin(ua_df, how="left", predicate='intersects')
#users_codes

In [ ]:
# merge to all users
ua_users = pd.merge(
    left=users, 
    right=users_codes[['id', 'year', 'district_en']], 
    left_on=['id', 'year'], right_on = ['id', 'year'],
    how='left'
)
ua_users

In [ ]:
# fill missing districts using location data
loc= 'kyiv|kiyv|kiev|київ|киев'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Misto Kyiv'

loc= 'kharkiv|kharkov|харків|харьков'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Kharkivska oblast'

loc= 'odessa|odesa|одесса|одеса|odes'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Odeska oblast'

loc= 'dnipropetrovsk|dnepropetrovsk|дніпропетровськ|днепропетровск|dnipro|dnepr|днепро|дніпро|kryvyi rih|krivo|кривой|кривий'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Dnipropetrovska oblast'

loc= 'donet|донецк|донецьк|mariupol|маріупол|мариуполь|макіївка|макиевка|makievka|makiivka|horlivka|gorlovka|горловка'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Donetska oblast'

loc= 'zapoje|zaporizh|zaporozh|zaporizhzhya|zaporozhje|запорожье|запоріжжа|мелитополь|melitopol|berdiansk|бердян'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Zaporizka oblast'

loc= 'l\'viv|lviv|lvov|львів|львов'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Lvivska oblast'

loc= 'cherkasy|черкасы|cherkass'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Cherkaska oblast'

loc= 'vinnytsya|вінниця|винница|vinnytsia|vinni'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Vinnytska oblast'

loc= 'kherson|херсон'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Khersonska oblast'

loc= 'mykolaiv|миколаїв|николаев|nikolay|nikolae'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Mykolaivska oblast'

loc= 'luhansk|луганск'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Luhanska oblast'

loc= 'krym|simferopol|yalta|крым|симферополь|ялта|crimea|crim'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Avtonomna Respublika Krym'

loc= 'sevastopol|севастополь'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Sevastopol (miskrada)'

loc= 'chernihiv|chernigov|chernigiv|чернигов|чернігів'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Chernihivska oblast'

loc= 'poltav|полтав|kremenc|кременч'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Poltavska oblast'

loc= 'sumy|сумы'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Sumska oblast'

loc= 'kirovohrad|кіровоград|кировоград'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Kirovohradska oblast'

loc= 'ivano-frankivsk|ивано-франковск|iвано-франківськ|ivano-|івано-'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Ivano-Frankivska oblast'

loc= 'ternopil|тернопіль|тернополь'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Ternopilska oblast'

loc= 'uzhhorod|uzhgorod|ужгород|mukachev|mucache|мукачев|чоп'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Zakarpatska oblast'

loc= 'rivne|рівне|ровно|rovno'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Rivnenska oblast'

loc= 'khmeln|hmeln|хмельн'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Khmelnytska oblast'

loc= 'zhytomir|zhytomyr|житомир|zhitomir'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Zhytomyrska oblast'

loc= 'volyn|волынь|луцьк|луцк|lutsk'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Volynska oblast'

loc= 'chernivts|chernovt|черновцы|чернівц'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Chernivetska oblast'

loc= 'bucha|irpin|irpen|буча|ирпень|ірпінь|brovar|бровар|белая церков|bila tserk|біла церква'
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains(loc)), 'district_en'] = 'Kyivska oblast'

#no specific info
ua_users.loc[ua_users.loc_current.str.lower().str.strip() == 'ukraine', 'district_en'] = 'Ukraine'
ua_users.loc[ua_users.loc_current.str.lower().str.strip() == 'ua', 'district_en'] = 'Ukraine'

In [ ]:
pd.DataFrame(ua_users.loc[ua_users['district_en'].isnull(), 'loc_current'].value_counts())

# replace the rest missing with 'Ukraine' (except for UAE)
ua_users.loc[(ua_users['district_en'].isnull()) & (ua_users.loc_current.str.lower().str.contains('ukrain|укра')), 'district_en'] = 'Ukraine'

In [ ]:
pd.DataFrame(ua_users.loc[ua_users['district_en'].isnull(), 'loc_current'].value_counts())

In [ ]:
ua_users.district_en.value_counts()

In [ ]:
# Donetsk, Luhansk, Krym
dlk= 'donetsk|luhansk|krym|sevasto'
ua_users.loc[ua_users['district_en'].fillna('').str.lower().str.contains(dlk), 'dlk'] = 1
ua_users.loc[ua_users['dlk'].isnull(), 'dlk'] = 0
ua_users['dlk'].value_counts()

In [ ]:
# Identity proxy: share of russians < median (same for russian language usage)
ru_identity= 'donetsk|luhansk|krym|sevasto|dnipro|kharkiv|kherson|luhansk|mykolaiv|odesk|zaporiz'
ua_users.loc[ua_users['district_en'].fillna('').str.lower().str.contains(ru_identity), 'ru_identity'] = 1
ua_users.loc[ua_users['district_en'].fillna('').str.lower().str.contains('ukraine|misto'), 'ru_identity'] = 0.5 # Ukraine and Kyiv (not sure about kyiv however)
ua_users.loc[ua_users['ru_identity'].isnull(), 'ru_identity'] = 0
ua_users['ru_identity'].value_counts()

In [ ]:
ua_users[ua_users['ru_identity']==1].district_en.value_counts()

In [ ]:
ua_users[ua_users['ru_identity']==0].district_en.value_counts()

In [ ]:
# name identity
n = 'cyril|anatoly|andy|vitalik|mark|andre|john|eduard|daniel|valerii|stepan|timur|artyom|roma|peter|petr|maks|den|arthur|stas|anatoliy|mike|serge|paul|nick|vitaly|vitali|valik|viktor|ruslan|victor|vlad|yaroslav|vitaliy|eugene|denis|artem|ivan|andrew|anton|alex|roman|lee|alec|eldar|serhio|steve|harry|nicholas|artiom|seva|tony|denis|dennis|kir|steven|ulia|anastasi|natali|natalli|nataly|kate|katya|oksana|julia|yulia|anastasi|natalia|anna|inna|mariia|edward|ed|lyudmila|maria|marija|mariya|maryia|anna|nina|petr|lyubov|ljubov|kate|raisa|tamara|nastia|alla|sofia|денис|axel|masha|andru|ksenia|артур|arkad|iaroslav|ignat|sergio|george|georgii|georgey|serge|jeka|артем|антон|leo|joseph|vsevolod|yarik|yuliia|raul|роман|nata|iuliia|rita|robert|romeo|juli|jack|jonathan|marta|chris|philip|arthur|george|vika|helen|max|mary|martin|zakhar|zhenya|tim|timofey|serj|stan|stanislaw|svyatoslav|serveryn|sir|marat|liubov|nastya|nik|rustam|serg|sam'
ua_users.loc[ua_users['name2013'].fillna('').str.lower().str.contains(n), 'ru_name2013'] = 0.5 #undefined
ua_users.loc[ua_users['name2016'].fillna('').str.lower().str.contains(n), 'ru_name2016'] = 0.5 #undefined

n = 'slavko|andrij|herman|andry|ostap|sergij|serhiy|fedir|khrystyna|eugen|mykhaylo|myroslav|olexiy|kyrylo|danylo|arsen|yevhen|borys|valentyn|evgen|kostiantyn|kostiantin|ievgen|petro|olexandr|nazar|serhii|mykhailo|oleksiy|vladyslav|vadym|mykola|oleksii|serhiy|ihor|oleh|vasyl|sergeii|oleks|demian|pavlo|maksym|bohdan|yury|sergiy|andriy|kyiv|taras|volodym|volodim|dmytro|dmitro|marko|evhenii|hennadii|ukrainian|grygori|myr|valery|illia|tymur|illya|yevgen|yevhen|mykyta|mikita|maxym|maksym|olexi|olexy|hlib|glib|myron|rostyslav|oles|andrian|lyubomyr|iryna|maryna|kateryna|svitlana|vira|nadiya|nadya|nadiia|hanna|ivanna|myroslava|tetiana|olexander|valentyna|darya|darina|daryna|olesia|yana|jana|maryana|olha|olena|tetyana|halina|oleksandra|viktorya|henadi|hennagi|ganna|lesya|michal|yurko|andrii|sergii|serhij|олекс|сергій|володимир|володімір|микита|михайло|ігор|павло|дмитро|петро|данило|тарас|назар|мар\'ян|є|і|дарина|тетяна|ганна|олен|яна'
ua_users.loc[ua_users['name2013'].fillna('').str.lower().str.contains(n), 'ru_name2013'] = 0 #UA name
ua_users.loc[ua_users['name2016'].fillna('').str.lower().str.contains(n), 'ru_name2016'] = 0 #UA name

n = 'aleks|alekse|sergei|sergey|nikola|iakov|nikolai|aleksand|ilya|yevgeny|egor|orest|david|dmitrij|gennadiy|rodion|alona|alena|aljona|dmytry|pasha|nickolai|андрей|валерий|serhey|yevhenii|nicolai|sviatoslav|aleskander|fedor|yegor|yurij|constantine|danil|illia|viacheslav|michail|volodia|miroslav|iurii|yuri|yura|jurij|valeriy|nikita|mikhail|evgeniy|misha|sasha|kostya|valentin|vasiliy|vova|sergei|sergej|sergey|mihail|leonid|slava|yura|vyacheslav|gennady|slavik|maksim|andrei|kostia|konstantyn|konstantin|michael|vladislav|stanislav|vadim|boris|bogdan|dima|aleksandr|maxim|dmitriy|yuriy|alexey|oleg|vladim|vanya|vasily|vasya|hleb|igor|rostislav|gleb|pavel|ilia|illarion|dmitry|dimon|dimitriy|dimitry|dmitri|dmitrii|daniil|andrey|sergey|alexander|rinat|filip|constantin|kiril|kolya|miron|irina|marina|elena|olga|valentina|galina|tatiana|lena|ekaterina|vera|victoria|viktoria|svetlana|daria|alyona|tania|tanya|tolik|константин|николай|владислав|станислав|иван|евгений|алекс|сергей|владимир|никита|михаил|игорь|павел|дмитрий|петр|даниил|илья|татьяна|елена|алена|софия|наталия|виктория|виктор|виталий|вера|надежда|геннадий|кирилл|юрий'
ua_users.loc[ua_users['name2013'].fillna('').str.lower().str.contains(n), 'ru_name2013'] = 1 #RU name
ua_users.loc[ua_users['name2016'].fillna('').str.lower().str.contains(n), 'ru_name2016'] = 1 #RU name


In [ ]:
ua_users.ru_name2013.value_counts()

In [ ]:
ua_users.ru_name2016.value_counts()

In [ ]:
ua_data.head()

In [ ]:
ua = pd.merge(
    left=ua_users[['id', 'year', 'district_en', 'dlk', 'ru_identity', 'ru_name2013', 'ru_name2016' ]],
    right=ua_data[['district_en', 'code', 'code_geo', 'pop_T', 'russians', 'language_ru_T']],
    left_on='district_en', right_on = 'district_en',
    how='left'
)
ua

In [ ]:
len(ua[ua.district_en.isnull()])/len(ua)

In [ ]:
ua.to_csv(EXTRACTIONS / 'ua.csv', encoding = 'utf-8', index=False)

In [ ]:
# change to google cloud kernel 
import os, sys
from google.cloud import bigquery
from google.cloud import storage

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# upload to google storage 
bucket_name =  'github_user_datasets'
destination_blob_name = 'ua_users.csv'
source_file_name = EXTRACTIONS / 'ua.csv'

bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(source_file_name)

print(
        f"File {source_file_name} uploaded to {destination_blob_name}."
)
    

In [ ]:
# load a table from GS to bigquery
table_id = "friends-foes.user_panel.ua_users_new"
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id", "INTEGER"),
        bigquery.SchemaField("year", "INTEGER"),
        bigquery.SchemaField("district_en", "STRING"),
        bigquery.SchemaField("dlk", "FLOAT"),
        bigquery.SchemaField("ru_identity", "FLOAT"),
        bigquery.SchemaField("ru_name2013", "FLOAT"),
        bigquery.SchemaField("ru_name2016", "FLOAT"),
        bigquery.SchemaField("code", "FLOAT"),
        bigquery.SchemaField("code_geo", "FLOAT"),
        bigquery.SchemaField("population", "FLOAT"),
        bigquery.SchemaField("russians", "FLOAT"),
        bigquery.SchemaField("language_ru", "FLOAT")       
    ],
    skip_leading_rows=1
    )

uri = "gs://github_user_datasets/ua_users.csv"

load_job = client.load_table_from_uri(
    uri, table_id, job_config=job_config
)  # Make an API request.

load_job.result()  # Wait for the job to complete.

table = client.get_table(table_id)
print("Loaded {} rows to table {}".format(table.num_rows, table_id))